# 05 - Modelos de clasificación

Entrenamos tres modelos en orden de complejidad creciente y comparamos
su performance usando métricas apropiadas para datasets desbalanceados.

## 1. Carga de datos procesados

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import cross_val_score
from xgboost import XGBClassifier
import joblib

X_train = pd.read_csv('../data/X_train.csv')
X_test = pd.read_csv('../data/X_test.csv')
y_train = pd.read_csv('../data/y_train.csv').squeeze()
y_test = pd.read_csv('../data/y_test.csv').squeeze()

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")

X_train: (9420, 4)
X_test:  (2355, 4)
y_train: (9420,)
y_test:  (2355,)


## 2. Modelo 1 — Regresión Logística (baseline)

El modelo más simple de clasificación binaria. Nos da un piso de 
performance contra el cual comparar los modelos más complejos.

class_weight='balanced' le indica al modelo que los asteroides 
peligrosos pesan más en el aprendizaje, compensando el desbalance 
de clases que detectamos en el EDA.

In [3]:
lr = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

print("Regresión Logística")
print(classification_report(y_test, y_pred_lr, target_names=['No peligroso', 'Peligroso']))

Regresión Logística
              precision    recall  f1-score   support

No peligroso       1.00      0.89      0.94      2243
   Peligroso       0.31      1.00      0.47       112

    accuracy                           0.89      2355
   macro avg       0.65      0.94      0.70      2355
weighted avg       0.97      0.89      0.92      2355



## 3. Modelo 2 — Random Forest

In [4]:
rf = RandomForestClassifier(class_weight='balanced', random_state=42, n_estimators=100)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("Random Forest")
print(classification_report(y_test, y_pred_rf, target_names=['No peligroso', 'Peligroso']))

Random Forest
              precision    recall  f1-score   support

No peligroso       0.97      0.99      0.98      2243
   Peligroso       0.74      0.33      0.46       112

    accuracy                           0.96      2355
   macro avg       0.85      0.66      0.72      2355
weighted avg       0.96      0.96      0.96      2355



## 4. Modelo 3 — XGBoost

In [6]:
scale_pos_weight = y_train.value_counts()[False] / y_train.value_counts()[True]

xgb = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss'
)
xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)

print("XGBoost")
print(classification_report(y_test, y_pred_xgb, target_names=['No peligroso', 'Peligroso']))

XGBoost
              precision    recall  f1-score   support

No peligroso       0.98      0.97      0.98      2243
   Peligroso       0.50      0.68      0.58       112

    accuracy                           0.95      2355
   macro avg       0.74      0.82      0.78      2355
weighted avg       0.96      0.95      0.96      2355

